In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
# Configuration dynamique
DATA_DIR = Path('capture/train_data/Prise mouvement')
movement_dirs = sorted(
    [d for d in DATA_DIR.iterdir() if d.is_dir() and d.name.startswith('mouvement_')],
    key=lambda d: int(d.name.split('_')[1]),
)
NUM_CLASSES = len(movement_dirs)


In [ ]:
# Charger les données
X_data = []
y_data = []

for movement_dir in movement_dirs:
    label = int(movement_dir.name.split('_')[1]) - 1  # 0-indexé
    npy_files = sorted(movement_dir.glob(f"{movement_dir.name}_*.npy"))
    for f in npy_files:
        X_data.append(np.load(f))
        y_data.append(label)

print(f"{NUM_CLASSES} mouvements | {len(X_data)} séquences")

In [ ]:
import os

# Labels auto-détectés depuis les noms de dossiers
labels = [d.name for d in movement_dirs]

print(f"Classes: {NUM_CLASSES}")

In [ ]:
label_map = {label:num for num, label in enumerate(labels)}

In [ ]:
label_map

In [ ]:
MAX_LEN = 60

X_padded = pad_sequences(
    X_data,
    maxlen=MAX_LEN,
    dtype="float32",
    padding="post",
    truncating="post",
    value=0.0,
)

# Normalisation : mean=0, std=1
mean = X_padded[X_padded != 0.0].mean()
std = X_padded[X_padded != 0.0].std()
X_padded = np.where(X_padded != 0.0, (X_padded - mean) / std, 0.0)

print(f"Shape: {X_padded.shape}, Mean: {X_padded[X_padded != 0.0].mean():.4f}, Std: {X_padded[X_padded != 0.0].std():.4f}")

In [ ]:
len(X_padded)

In [ ]:
y = to_categorical(y_data, num_classes=NUM_CLASSES).astype(int)
y

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_padded, y,
    test_size=0.2,
    stratify=y_data,
    random_state=69,
)


In [ ]:
y_train.shape

X_train.shape

In [ ]:
# Neural Network Architecture
model = Sequential()
model.add(layers.Input(shape=(60, 282)))
model.add(layers.LSTM(64, return_sequences=True))
model.add(layers.LSTM(64, return_sequences=False))
model.add(layers.Dropout(0.3))
model.add(layers.Dense(32, activation='relu'))
model.add(layers.Dropout(0.3))
model.add(layers.Dense(NUM_CLASSES, activation='softmax'))

model.summary()

In [ ]:

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5),
]

history = model.fit(
    X_train, y_train,
    epochs=100,
    validation_data=(X_test, y_test),
    callbacks=callbacks,
)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


df_hist = pd.DataFrame(history.history)
df_hist['epoch'] = range(1, len(df_hist) + 1)
df_hist = df_hist.melt(id_vars='epoch', var_name='metric', value_name='value')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(data=df_hist[df_hist['metric'].str.contains('accuracy')], x='epoch', y='value', hue='metric', ax=ax1)
ax1.set_title('Accuracy')

sns.lineplot(data=df_hist[df_hist['metric'].str.contains('loss')], x='epoch', y='value', hue='metric', ax=ax2)
ax2.set_title('Loss')

plt.show()

## Analyse de features

In [ ]:
# Variance par feature sur l'ensemble du dataset
var_per_feature = X_padded.reshape(-1, 282).var(axis=0)

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(282), var_per_feature, width=1)
ax.axvline(132, color='r', linestyle='--', label='Face start')
ax.axvline(156, color='g', linestyle='--', label='Left hand start')
ax.axvline(219, color='b', linestyle='--', label='Right hand start')
ax.set_title('Variance par feature (0-131=Pose, 132-155=Face, 156-218=Left hand, 219-281=Right hand)')
ax.set_xlabel('Feature index')
ax.set_ylabel('Variance')
ax.legend()
plt.tight_layout()
plt.show()

# Features avec variance quasi nulle (inutiles)
low_var = np.where(var_per_feature < 0.01)[0]
print(f"Features quasi constantes (var < 0.01) : {len(low_var)} / 282")
print(f"Indices : {low_var}")

In [ ]:
# Heatmap des corrélations entre les 4 groupes de features
import matplotlib.pyplot as plt

pose = X_padded[:, :, :132].reshape(len(X_padded), -1)
face = X_padded[:, :, 132:156].reshape(len(X_padded), -1)
left_hand = X_padded[:, :, 156:219].reshape(len(X_padded), -1)
right_hand = X_padded[:, :, 219:282].reshape(len(X_padded), -1)

groups = {
    'Pose': pose,
    'Face': face,
    'Left Hand': left_hand,
    'Right Hand': right_hand,
}

# Matrice de corrélation entre groupes
corr_matrix = pd.DataFrame({
    name: {other: np.corrcoef(g1.mean(axis=1), g2.mean(axis=1))[0, 1]
           for other, g2 in groups.items()}
    for name, g1 in groups.items()
}).astype(float)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, ax=ax1,
            vmin=-1, vmax=1, fmt='.2f')
ax1.set_title('Corrélation entre groupes de features')


In [ ]:
# Retirer les features constantes
var_per_feature = X_padded.reshape(-1, 282).var(axis=0)
useful_mask = var_per_feature >= 0.01

print(f"Features utiles : {useful_mask.sum()} / 282")
print(f"Features retirées : {(~useful_mask).sum()}")

# Filtrer
X_clean = X_padded[:, :, useful_mask]
print(f"Nouvelle shape : {X_clean.shape}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y,
    test_size=0.2,
    stratify=y_data,
    random_state=69,
)

In [ ]:
model = Sequential()
model.add(layers.Input(shape=X_train.shape[1:]))
model.add(layers.LSTM(64, return_sequences=True))
model.add(layers.LSTM(64, return_sequences=False))
model.add(layers.Dropout(0.3))
model.add(layers.Dense(32, activation='relu'))
model.add(layers.Dropout(0.3))
model.add(layers.Dense(NUM_CLASSES, activation='softmax'))

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5),
]

history = model.fit(
    X_train, y_train,
    epochs=100,
    validation_data=(X_test, y_test),
    callbacks=callbacks,
)

## Data Augmentation


In [ ]:
def augment_mirror(X, y):
    """Double le dataset par miroir horizontal (gauche↔droite)."""
    structure = {
        'pose':       (33, 4),  # x,y,z,visibility
        'face':       (8,  3),  # x,y,z
        'left_hand':  (21, 3),  # x,y,z
        'right_hand': (21, 3),  # x,y,z
    }

    slices = {}
    offset = 0
    for name, (count, dims) in structure.items():
        size = count * dims
        slices[name] = (slice(offset, offset + size), count, dims)
        offset += size

    x_indices = []
    for name, (slc, count, dims) in slices.items():
        x_indices.extend(range(slc.start, slc.stop, dims))

    X_mirror = X.copy()

    # Inverser l'axe x
    X_mirror[:, :, x_indices] = 1.0 - X_mirror[:, :, x_indices]

    # Échanger main gauche ↔ droite
    lh = slices['left_hand'][0]
    rh = slices['right_hand'][0]
    X_mirror[:, :, lh], X_mirror[:, :, rh] = (
        X_mirror[:, :, rh].copy(),
        X_mirror[:, :, lh].copy(),
    )

    # Échanger pose gauche ↔ droite (indices 11-32, pairs=gauche, impairs=droite)
    pose_slc, _, pose_dims = slices['pose']
    for i in range(11, 33, 2):
        l = slice(pose_slc.start + i * pose_dims, pose_slc.start + (i+1) * pose_dims)
        r = slice(pose_slc.start + (i+1) * pose_dims, pose_slc.start + (i+2) * pose_dims)
        X_mirror[:, :, l], X_mirror[:, :, r] = (
            X_mirror[:, :, r].copy(),
            X_mirror[:, :, l].copy(),
        )

    X_aug = np.concatenate([X, X_mirror], axis=0)
    y_aug = np.concatenate([y, y], axis=0)

    print(f"Avant: {len(X)} | Après: {len(X_aug)}")
    return X_aug, y_aug

In [ ]:
y = to_categorical(y_data, num_classes=NUM_CLASSES).astype(int)
X_aug, y_aug = augment_mirror(X_padded, y)

X_train, X_test, y_train, y_test = train_test_split(
    X_aug, y_aug,
    test_size=0.2,
    stratify=np.argmax(y_aug, axis=1),
    random_state=69,
)
print(f"Train: {len(X_train)} | Test: {len(X_test)} | Classes: {NUM_CLASSES} | Features: {X_aug.shape[2]}")

In [ ]:
model_aug = Sequential()
model_aug.add(layers.Input(shape=X_train.shape[1:]))
model_aug.add(layers.LSTM(64, return_sequences=True))
model_aug.add(layers.LSTM(64, return_sequences=False))
model_aug.add(layers.Dropout(0.3))
model_aug.add(layers.Dense(32, activation='relu'))
model_aug.add(layers.Dropout(0.3))
model_aug.add(layers.Dense(NUM_CLASSES, activation='softmax'))

model_aug.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

model_aug.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5),
]

history = model_aug.fit(
    X_train, y_train,
    epochs=100,
    validation_data=(X_test, y_test),
    callbacks=callbacks,
)

In [ ]:
df_hist = pd.DataFrame(history.history)
df_hist['epoch'] = range(1, len(df_hist) + 1)
df_hist = df_hist.melt(id_vars='epoch', var_name='metric', value_name='value')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(data=df_hist[df_hist['metric'].str.contains('accuracy')], x='epoch', y='value', hue='metric', ax=ax1)
ax1.set_title('Accuracy')

sns.lineplot(data=df_hist[df_hist['metric'].str.contains('loss')], x='epoch', y='value', hue='metric', ax=ax2)
ax2.set_title('Loss')

plt.show()

## Vitesse 


In [ ]:
def add_velocity(X):
    """Ajoute les features de vitesse (différence frame-to-frame)."""
    velocity = np.zeros_like(X)
    velocity[:, 1:, :] = X[:, 1:, :] - X[:, :-1, :]

    padding_mask = (X == 0.0).all(axis=2, keepdims=True)
    velocity = np.where(padding_mask, 0.0, velocity)

    X_with_vel = np.concatenate([X, velocity], axis=2)
    print(f"Features: {X.shape[2]} → {X_with_vel.shape[2]} (positions + vitesses)")
    return X_with_vel

In [ ]:
X_vel = add_velocity(X_padded)

In [ ]:
y = to_categorical(y_data, num_classes=NUM_CLASSES).astype(int)
X_aug, y_aug = augment_mirror(X_vel, y)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_aug, y_aug,
    test_size=0.2,
    stratify=np.argmax(y_aug, axis=1),
    random_state=69,
)
print(f"Train: {len(X_train)} | Test: {len(X_test)} | Classes: {NUM_CLASSES}")

In [ ]:
model = Sequential()
model.add(layers.Input(shape=X_train.shape[1:]))
model.add(layers.LSTM(64, return_sequences=True))
model.add(layers.LSTM(64, return_sequences=False))
model.add(layers.Dropout(0.3))
model.add(layers.Dense(32, activation='relu'))
model.add(layers.Dropout(0.3))
model.add(layers.Dense(NUM_CLASSES, activation='softmax'))

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5),
]

history = model.fit(
    X_train, y_train,
    epochs=100,
    validation_data=(X_test, y_test),
    callbacks=callbacks,
)

In [ ]:
df_hist = pd.DataFrame(history.history)
df_hist['epoch'] = range(1, len(df_hist) + 1)
df_hist = df_hist.melt(id_vars='epoch', var_name='metric', value_name='value')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(data=df_hist[df_hist['metric'].str.contains('accuracy')], x='epoch', y='value', hue='metric', ax=ax1)
ax1.set_title('Accuracy')

sns.lineplot(data=df_hist[df_hist['metric'].str.contains('loss')], x='epoch', y='value', hue='metric', ax=ax2)
ax2.set_title('Loss')

plt.show()

## Accélération


In [ ]:
def add_kinematics(X):
    """Ajoute les features de vitesse et d'accélération."""
    # Vitesse : frame[t] - frame[t-1]
    velocity = np.zeros_like(X)
    velocity[:, 1:, :] = X[:, 1:, :] - X[:, :-1, :]

    # Accélération : velocity[t] - velocity[t-1]
    acceleration = np.zeros_like(X)
    acceleration[:, 2:, :] = velocity[:, 2:, :] - velocity[:, 1:-1, :]

    # Zéro sur le padding
    padding_mask = (X == 0.0).all(axis=2, keepdims=True)
    velocity = np.where(padding_mask, 0.0, velocity)
    acceleration = np.where(padding_mask, 0.0, acceleration)

    X_full = np.concatenate([X, velocity, acceleration], axis=2)
    print(f"Features: {X.shape[2]} → {X_full.shape[2]} (positions + vitesses + accélérations)")
    return X_full

In [ ]:
X_kine = add_kinematics(X_padded)

In [ ]:
y = to_categorical(y_data, num_classes=NUM_CLASSES).astype(int)
X_aug, y_aug = augment_mirror(X_kine, y)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_aug, y_aug,
    test_size=0.2,
    stratify=np.argmax(y_aug, axis=1),
    random_state=69,
)
print(f"Train: {len(X_train)} | Test: {len(X_test)} | Classes: {NUM_CLASSES}")

In [ ]:
model = Sequential()
model.add(layers.Input(shape=X_train.shape[1:]))
model.add(layers.LSTM(64, return_sequences=True))
model.add(layers.LSTM(64, return_sequences=False))
model.add(layers.Dropout(0.3))
model.add(layers.Dense(32, activation='relu'))
model.add(layers.Dropout(0.3))
model.add(layers.Dense(NUM_CLASSES, activation='softmax'))

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5),
]

history = model.fit(
    X_train, y_train,
    epochs=100,
    validation_data=(X_test, y_test),
    callbacks=callbacks,
)

In [ ]:
df_hist = pd.DataFrame(history.history)
df_hist['epoch'] = range(1, len(df_hist) + 1)
df_hist = df_hist.melt(id_vars='epoch', var_name='metric', value_name='value')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(data=df_hist[df_hist['metric'].str.contains('accuracy')], x='epoch', y='value', hue='metric', ax=ax1)
ax1.set_title('Accuracy')

sns.lineplot(data=df_hist[df_hist['metric'].str.contains('loss')], x='epoch', y='value', hue='metric', ax=ax2)
ax2.set_title('Loss')

plt.show()

## Analyse

In [ ]:
# Détail par bloc : combien de features utiles vs constantes
X_full = add_kinematics(X_padded)  # 846 features
var_all = X_full.reshape(-1, X_full.shape[2]).var(axis=0)

print(f"{'Section':<15} {'Total':>6} {'Utiles':>6} {'Constantes':>10} {'% utile':>8}")
print("-" * 50)

for block_idx, label in [(0, 'Positions'), (1, 'Vitesses'), (2, 'Accélérations')]:
    start = block_idx * 282
    end = start + 282
    total = 282
    constant = np.sum(var_all[start:end] < 0.01)
    useful = total - constant
    pct = useful / total * 100
    print(f"{label:<15} {total:>6} {useful:>6} {constant:>10} {pct:>7.1f}%")

In [ ]:
# Filtrer les features constantes sur les données cinématiques
X_full = add_kinematics(X_padded)
var_all = X_full.reshape(-1, X_full.shape[2]).var(axis=0)
useful_mask = var_all >= 0.01

X_filtered = X_full[:, :, useful_mask]
print(f"846 → {X_filtered.shape[2]} features utiles")

y = to_categorical(y_data, num_classes=NUM_CLASSES).astype(int)
X_aug, y_aug = augment_mirror(X_filtered, y)

X_train_filtered, X_test_filtered, y_train_filtered, y_test_filtered = train_test_split(
    X_aug, y_aug,
    test_size=0.2,
    stratify=np.argmax(y_aug, axis=1),
    random_state=69,
)
print(f"Train: {len(X_train)} | Test: {len(X_test)} | Features: {X_filtered.shape[2]}")

In [ ]:
model_filtered = Sequential()
model_filtered.add(layers.Input(shape=X_train.shape[1:]))
model_filtered.add(layers.LSTM(64, return_sequences=True))
model_filtered.add(layers.LSTM(64, return_sequences=False))
model_filtered.add(layers.Dropout(0.3))
model_filtered.add(layers.Dense(32, activation='relu'))
model_filtered.add(layers.Dropout(0.3))
model_filtered.add(layers.Dense(NUM_CLASSES, activation='softmax'))

model_filtered.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5),
]

history_filtered = model_filtered.fit(
    X_train, y_train,
    epochs=100,
    validation_data=(X_test, y_test),
    callbacks=callbacks,)

In [ ]:
df_hist = pd.DataFrame(history_filtered.history)
df_hist['epoch'] = range(1, len(df_hist) + 1)
df_hist = df_hist.melt(id_vars='epoch', var_name='metric', value_name='value')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(data=df_hist[df_hist['metric'].str.contains('accuracy')], x='epoch', y='value', hue='metric', ax=ax1)
ax1.set_title('Accuracy')

sns.lineplot(data=df_hist[df_hist['metric'].str.contains('loss')], x='epoch', y='value', hue='metric', ax=ax2)
ax2.set_title('Loss')

plt.show()

In [ ]:
X_full = add_kinematics(X_padded)
var_all = X_full.reshape(-1, X_full.shape[2]).var(axis=0)

for threshold in [0.01, 0.005, 0.003, 0.001]:
    kept = np.sum(var_all >= threshold)
    dropped = np.sum(var_all < threshold)
    print(f"Seuil {threshold:.3f} → {kept} utiles / {dropped} retirées")

## Réduction de l'over fitting
-> on filtre moins les features threshold à 0.003
-> on augment le dropout

In [ ]:
useful_mask = var_all >= 0.00075
X_filtered = X_full[:, :, useful_mask]
print(f"846 → {X_filtered.shape[2]} features")
# Pipeline
y = to_categorical(y_data, num_classes=NUM_CLASSES).astype(int)
X_aug, y_aug = augment_mirror(X_filtered, y)

X_train, X_test, y_train, y_test = train_test_split(
    X_aug, y_aug,
    test_size=0.2,
    stratify=np.argmax(y_aug, axis=1),
    random_state=69,
)
# Modèle avec + de régularisation
model_optimize1 = Sequential()
model_optimize1.add(layers.Input(shape=X_train.shape[1:]))
model_optimize1.add(layers.LSTM(64, return_sequences=True))
model_optimize1.add(layers.LSTM(64, return_sequences=False))
model_optimize1.add(layers.Dropout(0.3))       # 0.3 → 0.5
model_optimize1.add(layers.Dense(32, activation='relu'))
model_optimize1.add(layers.Dropout(0.3))       # 0.3 → 0.5
model_optimize1.add(layers.Dense(NUM_CLASSES, activation='softmax'))

model_optimize1.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5),
]

history_optimize1 = model_optimize1.fit(
    X_train, y_train,
    epochs=100,
    validation_data=(X_test, y_test),
    callbacks=callbacks,
)

In [ ]:
df_hist = pd.DataFrame(history_optimize1.history)
df_hist['epoch'] = range(1, len(df_hist) + 1)
df_hist = df_hist.melt(id_vars='epoch', var_name='metric', value_name='value')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(data=df_hist[df_hist['metric'].str.contains('accuracy')], x='epoch', y='value', hue='metric', ax=ax1)
ax1.set_title('Accuracy')

sns.lineplot(data=df_hist[df_hist['metric'].str.contains('loss')], x='epoch', y='value', hue='metric', ax=ax2)
ax2.set_title('Loss')

plt.show()

## on change de lstm


In [ ]:
useful_mask = var_all >= 0.01
X_vel = add_velocity(X_padded)
#X_filtered = X_full[:, :, useful_mask]
print(f"846 → {X_filtered.shape[2]} features")
# Pipeline
y = to_categorical(y_data, num_classes=NUM_CLASSES).astype(int)
X_aug, y_aug = augment_mirror(X_vel, y)

X_train, X_test, y_train, y_test = train_test_split(
    X_aug, y_aug,
    test_size=0.2,
    stratify=np.argmax(y_aug, axis=1),
    random_state=69,
)
# Modèle avec + de régularisation
model_optimize2 = Sequential()

model_optimize2.add(layers.Input(shape=X_train.shape[1:]))
model_optimize2.add(layers.Bidirectional(layers.LSTM(64, return_sequences=True)))
model_optimize2.add(layers.Bidirectional(layers.LSTM(64, return_sequences=False)))
model_optimize2.add(layers.Dropout(0.3))
model_optimize2.add(layers.Dense(32, activation='relu'))
model_optimize2.add(layers.Dropout(0.3))
model_optimize2.add(layers.Dense(NUM_CLASSES, activation='softmax'))

model_optimize2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

model_optimize2.summary()

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5),
]

history_optimize2 = model_optimize2.fit(
    X_train, y_train,
    epochs=100,
    validation_data=(X_test, y_test),
    callbacks=callbacks,
)

In [ ]:
df_hist = pd.DataFrame(history_optimize2.history)
df_hist['epoch'] = range(1, len(df_hist) + 1)
df_hist = df_hist.melt(id_vars='epoch', var_name='metric', value_name='value')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(data=df_hist[df_hist['metric'].str.contains('accuracy')], x='epoch', y='value', hue='metric', ax=ax1)
ax1.set_title('Accuracy')

sns.lineplot(data=df_hist[df_hist['metric'].str.contains('loss')], x='epoch', y='value', hue='metric', ax=ax2)
ax2.set_title('Loss')

plt.show()

In [ ]:
def augment_temporal(X, y, jitter_scale=0.02, warp_range=(0.8, 1.2), dropout_rate=0.05):
    """Augmentation temporelle : jitter + time warp + frame dropout."""
    rng = np.random.default_rng(42)
    X_aug = X.copy()
    seq_len = X.shape[1]

    for i in range(len(X_aug)):
        # 1. Jitter : petit bruit sur les valeurs non-nulles
        mask = X_aug[i] != 0.0
        noise = rng.normal(0, jitter_scale, X_aug[i].shape).astype(np.float32)
        X_aug[i] = np.where(mask, X_aug[i] + noise, X_aug[i])

        # 2. Time warp : étirer ou comprimer la séquence active
        active = mask.any(axis=1).sum()
        if active > 4:
            factor = rng.uniform(*warp_range)
            new_len = max(4, int(active * factor))
            if new_len != active and new_len <= seq_len:
                indices = np.linspace(0, active - 1, new_len).astype(int)
                warped = X_aug[i][indices]
                X_aug[i] = np.zeros_like(X_aug[i])
                X_aug[i][:new_len] = warped[:new_len]

        # 3. Frame dropout : supprimer quelques frames
        drop = rng.random(seq_len) < dropout_rate
        X_aug[i][drop] = 0.0

    X_result = np.concatenate([X, X_aug], axis=0)
    y_result = np.concatenate([y, y], axis=0)

    print(f"Avant: {len(X)} | Après: {len(X_result)}")
    return X_result, y_result

In [ ]:
# Vitesses
X_vel = add_velocity(X_padded)
y = to_categorical(y_data, num_classes=NUM_CLASSES).astype(int)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_vel, y,
    test_size=0.2,
    stratify=np.argmax(y, axis=1),
    random_state=69,
)

# Miroir puis temporel
X_train_aug, y_train_aug = augment_mirror(X_train, y_train)
X_train_aug, y_train_aug = augment_temporal(X_train_aug, y_train_aug)



print(f"Train: {len(X_train)} | Test: {len(X_test)} | Features: {X_aug.shape[2]}")

model_optimize3 = Sequential()

model_optimize3.add(layers.Input(shape=X_train_aug.shape[1:]))
model_optimize3.add(layers.Bidirectional(layers.LSTM(64, return_sequences=True)))
model_optimize3.add(layers.Bidirectional(layers.LSTM(64, return_sequences=False)))
model_optimize3.add(layers.Dropout(0.3))
model_optimize3.add(layers.Dense(32, activation='relu'))
model_optimize3.add(layers.Dropout(0.3))
model_optimize3.add(layers.Dense(NUM_CLASSES, activation='softmax'))

model_optimize3.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)


callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5),
]

history_optimize3 = model_optimize3.fit(
    X_train_aug, y_train_aug,
    epochs=100,
    validation_data=(X_test, y_test),
    callbacks=callbacks,
)

In [ ]:
df_hist = pd.DataFrame(history_optimize3.history)
df_hist['epoch'] = range(1, len(df_hist) + 1)
df_hist = df_hist.melt(id_vars='epoch', var_name='metric', value_name='value')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(data=df_hist[df_hist['metric'].str.contains('accuracy')], x='epoch', y='value', hue='metric', ax=ax1)
ax1.set_title('Accuracy')

sns.lineplot(data=df_hist[df_hist['metric'].str.contains('loss')], x='epoch', y='value', hue='metric', ax=ax2)
ax2.set_title('Loss')

plt.show()


## Sauvegarde du meilleur modèle
Config optimale : Bidirectional LSTM + positions (282) + vitesses (282) = 564 features + mirror augmentation
Val accuracy : ~93.2%

In [ ]:
import json

# Sauvegarder le modèle
model_path = 'model_lsf_bilstm.keras'
model_optimize2.save(model_path)
print(f"✓ Modèle sauvegardé : {model_path}")

# Labels dynamiques
LABEL_NAMES = {
    1: "aider", 2: "ameliorer", 3: "ami", 4: "aujourd'hui",
    5: "bonjour", 6: "communiquer", 7: "entendant", 8: "content",
    9: "je_suis", 10: "je_veux", 11: "langue_des_signes", 12: "merci",
    13: "outil_pointage", 14: "outil", 15: "presenter", 16: "projet",
    17: "sourd_pointage", 18: "sourde",
}
labels = [LABEL_NAMES.get(int(d.name.split('_')[1]), d.name) for d in movement_dirs]

# Métadonnées
metadata = {
    "num_classes": NUM_CLASSES,
    "labels": labels,
    "input_shape": list(X_train.shape[1:]),
    "max_len": MAX_LEN,
    "features": "positions_282 + velocities_282 = 564",
    "normalization": {"mean": float(mean), "std": float(std)},
    "architecture": "Bidirectional LSTM (64x2) + Dense(32) + Dense(18)",
    "augmentation": "mirror",
    "val_accuracy": 0.932,
}

with open('model_lsf_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print(f"✓ Métadonnées sauvegardées : model_lsf_metadata.json")
print(f"  Classes : {NUM_CLASSES}")
print(f"  Input   : {metadata['input_shape']}")
print(f"  Labels  : {labels}")

## Ajouts features

In [1]:
def add_velocity(X):
    """Ajoute les features de vitesse (différence frame-to-frame)."""
    velocity = np.zeros_like(X)
    velocity[:, 1:, :] = X[:, 1:, :] - X[:, :-1, :]

    padding_mask = (X == 0.0).all(axis=2, keepdims=True)
    velocity = np.where(padding_mask, 0.0, velocity)

    X_with_vel = np.concatenate([X, velocity], axis=2)
    print(f"Features: {X.shape[2]} → {X_with_vel.shape[2]} (positions + vitesses)")
    return X_with_vel

In [2]:
def augment_mirror(X, y):
    """Double le dataset par miroir horizontal (gauche↔droite)."""
    structure = {
        'pose':       (33, 4),  # x,y,z,visibility
        'face':       (8,  3),  # x,y,z
        'left_hand':  (21, 3),  # x,y,z
        'right_hand': (21, 3),  # x,y,z
    }

    slices = {}
    offset = 0
    for name, (count, dims) in structure.items():
        size = count * dims
        slices[name] = (slice(offset, offset + size), count, dims)
        offset += size

    x_indices = []
    for name, (slc, count, dims) in slices.items():
        x_indices.extend(range(slc.start, slc.stop, dims))

    X_mirror = X.copy()

    # Inverser l'axe x
    X_mirror[:, :, x_indices] = 1.0 - X_mirror[:, :, x_indices]

    # Échanger main gauche ↔ droite
    lh = slices['left_hand'][0]
    rh = slices['right_hand'][0]
    X_mirror[:, :, lh], X_mirror[:, :, rh] = (
        X_mirror[:, :, rh].copy(),
        X_mirror[:, :, lh].copy(),
    )

    # Échanger pose gauche ↔ droite (indices 11-32, pairs=gauche, impairs=droite)
    pose_slc, _, pose_dims = slices['pose']
    for i in range(11, 33, 2):
        l = slice(pose_slc.start + i * pose_dims, pose_slc.start + (i+1) * pose_dims)
        r = slice(pose_slc.start + (i+1) * pose_dims, pose_slc.start + (i+2) * pose_dims)
        X_mirror[:, :, l], X_mirror[:, :, r] = (
            X_mirror[:, :, r].copy(),
            X_mirror[:, :, l].copy(),
        )

    X_aug = np.concatenate([X, X_mirror], axis=0)
    y_aug = np.concatenate([y, y], axis=0)

    print(f"Avant: {len(X)} | Après: {len(X_aug)}")
    return X_aug, y_aug

In [ ]:
LABEL_NAMES = {
    0: "aider", 1: "ameliorer", 2: "ami", 3: "aujourd'hui",
    4: "bonjour", 5: "communiquer", 6: "entendant", 7: "content",
    8: "je_suis", 9: "je_veux", 10: "langue_des_signes", 11: "merci",
    12: "outil_pointage", 13: "outil", 14: "presenter", 15: "projet",
    16: "sourd_pointage", 17: "sourde", 18: "vocal" , 19:  "traduction", 20: "",
}

X_vel = add_velocity(X_padded)
#X_filtered = X_full[:, :, useful_mask]
#print(f"846 → {X_filtered.shape[2]} features")
# Pipeline
y = to_categorical(y_data, num_classes=NUM_CLASSES).astype(int)
X_aug, y_aug = augment_mirror(X_vel, y)

X_train, X_test, y_train, y_test = train_test_split(
    X_aug, y_aug,
    test_size=0.2,
    stratify=np.argmax(y_aug, axis=1),
    random_state=69,
)
# Modèle avec + de régularisation
model_optimize2 = Sequential()

model_optimize2.add(layers.Input(shape=X_train.shape[1:]))
model_optimize2.add(layers.Bidirectional(layers.LSTM(64, return_sequences=True)))
model_optimize2.add(layers.Bidirectional(layers.LSTM(64, return_sequences=False)))
model_optimize2.add(layers.Dropout(0.3))
model_optimize2.add(layers.Dense(32, activation='relu'))
model_optimize2.add(layers.Dropout(0.3))
model_optimize2.add(layers.Dense(NUM_CLASSES, activation='softmax'))

model_optimize2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

model_optimize2.summary()

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5),
]

history_optimize2 = model_optimize2.fit(
    X_train, y_train,
    epochs=100,
    validation_data=(X_test, y_test),
    callbacks=callbacks,
)

In [ ]:
import json

# Sauvegarder le modèle
model_path = 'model_lsf_bilstm.keras'
model_optimize2.save(model_path)
print(f"✓ Modèle sauvegardé : {model_path}")

# Labels dynamiques
LABEL_NAMES = {
    1: "aider", 2: "ameliorer", 3: "ami", 4: "aujourd'hui",
    5: "bonjour", 6: "communiquer", 7: "entendant", 8: "content",
    9: "je_suis", 10: "je_veux", 11: "langue_des_signes", 12: "merci",
    13: "outil_pointage", 14: "outil", 15: "presenter", 16: "projet",
    17: "sourd_pointage", 18: "sourde", 19: "traduction", 20: "vocal", 21: "inconnu",
}
labels = [LABEL_NAMES.get(int(d.name.split('_')[1]), d.name) for d in movement_dirs]

# Métadonnées
metadata = {
    "num_classes": NUM_CLASSES,
    "labels": labels,
    "input_shape": list(X_train.shape[1:]),
    "max_len": MAX_LEN,
    "features": "positions_282 + velocities_282 = 564",
    "normalization": {"mean": float(mean), "std": float(std)},
    "architecture": "Bidirectional LSTM (64x2) + Dense(32) + Dense(18)",
    "augmentation": "mirror",
    "val_accuracy": 0.932,
}

with open('model_lsf_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print(f"✓ Métadonnées sauvegardées : model_lsf_metadata.json")
print(f"  Classes : {NUM_CLASSES}")
print(f"  Input   : {metadata['input_shape']}")
print(f"  Labels  : {labels}")

## Retrain sans mouvement_22 (STOP)
On garde 20 mouvements (0-19 = aider a vocal + 20 = inconnu), on retire mouvement_22.

In [5]:
from pathlib import Path
import numpy as np
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras import layers
import json

EXCLUDE = {'mouvement_22'}
DATA_DIR_20 = Path('capture/train_data/prompteur')
movement_dirs_20 = sorted(
    [d for d in DATA_DIR_20.iterdir() if d.is_dir() and d.name.startswith('mouvement_') and d.name not in EXCLUDE],
    key=lambda d: int(d.name.split('_')[1]),
)
NUM_CLASSES_20 = len(movement_dirs_20)

LABEL_NAMES = {
    1: "aider", 2: "ameliorer", 3: "ami", 4: "aujourd'hui",
    5: "bonjour", 6: "communiquer", 7: "entendant", 8: "content",
    9: "je_suis", 10: "je_veux", 11: "langue_des_signes", 12: "merci",
    13: "outil_pointage", 14: "outil", 15: "presenter", 16: "projet",
    17: "sourd_pointage", 18: "sourde", 19: "traduction", 20: "vocal", 21: "inconnu",
}

MAX_LEN = 60

X_data_20 = []
y_data_20 = []
for label_idx, movement_dir in enumerate(movement_dirs_20):
    npy_files = sorted(movement_dir.glob(f"{movement_dir.name}_*.npy"))
    for f in npy_files:
        X_data_20.append(np.load(f))
        y_data_20.append(label_idx)

print(f"{NUM_CLASSES_20} mouvements | {len(X_data_20)} sequences")

X_data_20 = [np.asarray(x, dtype=np.float32) for x in X_data_20]
X_padded_20 = pad_sequences(X_data_20, maxlen=MAX_LEN, dtype='float32', padding='post', truncating='post', value=0.0)
print(f"X_padded_20 shape: {X_padded_20.shape}")

mean_20 = X_padded_20[X_padded_20 != 0.0].mean()
std_20 = X_padded_20[X_padded_20 != 0.0].std()
X_padded_20 = np.where(X_padded_20 != 0.0, (X_padded_20 - mean_20) / std_20, 0.0)

X_vel_20 = add_velocity(X_padded_20)
y_20 = to_categorical(y_data_20, num_classes=NUM_CLASSES_20).astype(int)
X_aug_20, y_aug_20 = augment_mirror(X_vel_20, y_20)

X_train_20, X_test_20, y_train_20, y_test_20 = train_test_split(
    X_aug_20, y_aug_20,
    test_size=0.2,
    stratify=np.argmax(y_aug_20, axis=1),
    random_state=69,
)

print(f"Train: {X_train_20.shape} | Test: {X_test_20.shape}")

model_20 = Sequential()
model_20.add(layers.Input(shape=X_train_20.shape[1:]))
model_20.add(layers.Bidirectional(layers.LSTM(64, return_sequences=True)))
model_20.add(layers.Bidirectional(layers.LSTM(64, return_sequences=False)))
model_20.add(layers.Dropout(0.3))
model_20.add(layers.Dense(32, activation='relu'))
model_20.add(layers.Dropout(0.3))
model_20.add(layers.Dense(NUM_CLASSES_20, activation='softmax'))

model_20.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

model_20.summary()

callbacks_20 = [
    tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5),
]

history_20 = model_20.fit(
    X_train_20, y_train_20,
    epochs=100,
    validation_data=(X_test_20, y_test_20),
    callbacks=callbacks_20,
)

21 mouvements | 930 sequences
X_padded_20 shape: (930, 60, 282)
Features: 282 → 564 (positions + vitesses)
Avant: 930 | Après: 1860
Train: (1488, 60, 564) | Test: (372, 60, 564)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional_2 (Bidirectional) │ (None, 60, 128)        │       322,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 21)             │           693 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 425,685 (1.62 MB)

 Trainable params: 425,685 (1.62 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 94ms/step - accuracy: 0.0538 - loss: 3.0569 - val_accuracy: 0.0941 - val_loss: 3.0014 - learning_rate: 5.0000e-04
Epoch 2/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 4s 91ms/step - accuracy: 0.0867 - loss: 2.9861 - val_accuracy: 0.1237 - val_loss: 2.8927 - learning_rate: 5.0000e-04
Epoch 3/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 5s 112ms/step - accuracy: 0.1015 - loss: 2.8989 - val_accuracy: 0.1398 - val_loss: 2.7726 - learning_rate: 5.0000e-04
Epoch 4/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 5s 112ms/step - accuracy: 0.1391 - loss: 2.7681 - val_accuracy: 0.1613 - val_loss: 2.6717 - learning_rate: 5.0000e-04
Epoch 5/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 6s 119ms/step - accuracy: 0.1673 - loss: 2.6606 - val_accuracy: 0.2124 - val_loss: 2.5695 - learning_rate: 5.0000e-04
Epoch 6/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 7s 151ms/step - accuracy: 0.1888 - loss: 2.5911 - val_accuracy: 0.2097 - val_loss: 2.4911 - learning_rate: 5.0000e-04
Epoch 7/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 6s 128ms/step - accur

In [6]:
model_20.save('models/Model_VTH_20.keras')

labels_20 = [LABEL_NAMES.get(int(d.name.split('_')[1]), d.name) for d in movement_dirs_20]

metadata_20 = {
    'num_classes': NUM_CLASSES_20,
    'labels': labels_20,
    'input_shape': list(X_train_20.shape[1:]),
    'max_len': MAX_LEN,
    'features': 'positions_282 + velocities_282 = 564',
    'normalization': {'mean': float(mean_20), 'std': float(std_20)},
    'architecture': 'Bidirectional LSTM (64x2) + Dense(32) + Dense(20)',
    'augmentation': 'mirror',
}

with open('models/Model_VTH_20.json', 'w', encoding='utf-8') as f:
    json.dump(metadata_20, f, ensure_ascii=False, indent=2)

print(f'Modele sauvegarde : models/Model_VTH_20.keras')
print(f'Classes : {NUM_CLASSES_20}')
print(f'Labels  : {labels_20}')

Modele sauvegarde : models/Model_VTH_20.keras
Classes : 21
Labels  : ['aider', 'ameliorer', 'ami', "aujourd'hui", 'bonjour', 'communiquer', 'entendant', 'content', 'je_suis', 'je_veux', 'langue_des_signes', 'merci', 'outil_pointage', 'outil', 'presenter', 'projet', 'sourd_pointage', 'sourde', 'traduction', 'vocal', 'inconnu']
